# Library packages

In [5]:
library(dplyr)
library(survival)
library(plyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘survival’ was built under R version 4.4.3”
Warning message:
“package ‘plyr’ was built under R version 4.4.3”
------------------------------------------------------------------------------

You have loaded plyr after dplyr - this is likely to cause problems.
If you need functions from both plyr and dplyr, please load plyr first, then dplyr:
library(plyr); library(dplyr)

------------------------------------------------------------------------------


Attaching package: ‘plyr’


The following objects are masked from ‘package:dplyr’:

    arrange, count, desc, failwith, id, mutate, rename, summarise,
    summarize




In [ ]:
library(dplyr)
library(survival)
library(plyr)
library(tools) 

In [ ]:
cox_pipe <- function(data,
                     var_n,
                     filter_by_p_value = FALSE,  
                     p_cutoff = 0.05,           
                     covariate_order = NULL      
){
  
  colnames(data)[2] <- "status"
  colnames(data)[3] <- "time"
  
  surv <- Surv(time = data$time, event = data$status)
  data$surv <- surv

  default_vars <- colnames(data)[4:var_n]
  
  if (is.null(covariate_order)) {
    varNames <- default_vars
  } else {
    if (!all(covariate_order %in% default_vars)) {
      stop("covariate_order 中有不在数据协变量列里的名字，请检查。")
    }
    varNames <- covariate_order
  }

  UniCox <- function(x) {
    FML <- as.formula(paste0("surv ~ ", x))
    Cox <- coxph(FML, data = data)
    Sum <- summary(Cox)
    
    CI <- paste0(round(Sum$conf.int[, 3:4], 2), collapse = "-")
    Pvalue <- round(Sum$coefficients[, 5], 3)
    HR <- round(Sum$coefficients[, 2], 2)
    
    Unicox <- data.frame(
      "Characteristics" = x,
      "Hazard Ratio"    = HR,
      "CI95"            = CI,
      "P.value"         = Pvalue,
      stringsAsFactors  = FALSE
    )
    return(Unicox)
  }

  UniVar_list <- lapply(varNames, UniCox)
  UniVar <- ldply(UniVar_list, data.frame)

  multi_vars <- varNames
  if (filter_by_p_value) {
    sig_vars <- UniVar$Characteristics[!is.na(UniVar$P.value) &
                                         UniVar$P.value <= p_cutoff]
    if (length(sig_vars) == 0) {
      warning("没有单因素 p <", p_cutoff,
              " 的协变量，多因素 Cox 将不会拟合。")
      multi_vars <- character(0)
    } else {
      multi_vars <- sig_vars
    }
  }

  if (length(multi_vars) > 0) {
    fml <- as.formula(paste0("surv ~ ",
                             paste0(multi_vars, collapse = " + ")))
    multicox <- coxph(fml, data = data)
    multisum <- summary(multicox)
    
    multiname <- rownames(multisum$coefficients)  
    mCIL <- round(multisum$conf.int[, 3], 4)
    mCIU <- round(multisum$conf.int[, 4], 4)
    mCI <- paste0(mCIL, "-", mCIU)
    mPvalue <- round(multisum$coefficients[, 5], 4)
    mHR <- round(multisum$coefficients[, 2], 4)
    
    mulcox <- data.frame(
      "Characteristics"    = multiname,
      "Hazard Ratio.multi" = mHR,
      "CI95.multi"         = mCI,
      "P.value.multi"      = mPvalue,
      stringsAsFactors     = FALSE
    )
  } else {
    mulcox <- data.frame(
      "Characteristics"    = character(0),
      "Hazard Ratio.multi" = numeric(0),
      "CI95.multi"         = character(0),
      "P.value.multi"      = numeric(0),
      stringsAsFactors     = FALSE
    )
  }
  
  Final <- merge(UniVar, mulcox,
                 by = "Characteristics",
                 all = TRUE,
                 sort = FALSE) 
  Final$Characteristics <- factor(Final$Characteristics,
                                  levels = varNames)
  Final <- Final[order(Final$Characteristics), ]
  Final$Characteristics <- as.character(Final$Characteristics)
  
  return(Final)
}


# 上皮细胞的3个数据集csv输入

In [ ]:
epi_geo_meta <- read.csv('./Survival_analysis/Input/Cox_input_R3_Q8/Epi/GEO-meta/merged.uni_multi_cox_input.csv', stringsAsFactors = F)

In [ ]:
epi_gse26549 <- read.csv('./Survival_analysis/Input/Cox_input_R3_Q8/Epi/GSE26549/merged.uni_multi_cox_input.csv', stringsAsFactors = F)

In [ ]:
epi_tcga <- read.csv('./Survival_analysis/Input/Cox_input_R3_Q8/Epi/TCGA/merged.uni_multi_cox_input.csv', stringsAsFactors = F)

## GEO-meta

In [43]:
out_1 <- cox_pipe(epi_geo_meta, var_n = 10, filter_by_p_value = F, p_cutoff = 0.05)

In [44]:
out_1

,Characteristics,Hazard.Ratio,CI95,P.value,Hazard.Ratio.multi,CI95.multi,P.value.multi
,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,Stage_harmonized,1.61,1.22-2.11,0.001,1.6259,1.2309-2.1477,0.0006
2,Gender_harmonized,1.17,0.87-1.56,0.304,1.1099,0.8238-1.4955,0.4930
3,Age_harmonized,0.88,0.69-1.13,0.322,0.8734,0.6828-1.1173,0.2814
4,SBSN_Epi,0.77,0.6-0.98,0.030,0.8657,0.6784-1.1048,0.2464
5,COL17A1_Epi,1.69,1.29-2.22,0.000,1.8996,1.4393-2.507,0.0000
6,MKI67_Epi,1.57,1.21-2.03,0.001,1.5190,1.1619-1.9859,0.0022
7,GPX2_Epi,1.30,1.02-1.67,0.037,1.3079,1.0101-1.6936,0.0417


In [77]:
write.csv(out_1, './Survival_analysis/Output/R3-Q8/Epi_GEO-meta_COX_res.csv', row.names = FALSE, na = "")

## GSE26549

In [34]:
out_2 <- cox_pipe(epi_gse26549, var_n = 11, filter_by_p_value = F, p_cutoff = 0.05)

In [35]:
out_2

,Characteristics,Hazard.Ratio,CI95,P.value,Hazard.Ratio.multi,CI95.multi,P.value.multi
,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,Gender_harmonized,0.95,0.48-1.86,0.872,1.0146,0.4717-2.1824,0.9705
2,Age_harmonized,1.55,0.79-3.06,0.203,1.9014,0.9355-3.8643,0.0758
3,Smoking_harmonized,1.55,0.72-3.32,0.263,1.1744,0.4889-2.8209,0.7192
4,Alcohol_harmonized,1.03,0.5-2.11,0.935,0.8716,0.3839-1.9788,0.7425
5,MKI67_Epi,1.64,0.8-3.36,0.180,1.3832,0.641-2.985,0.4085
6,SBSN_Epi,1.25,0.63-2.49,0.519,1.6489,0.7571-3.5909,0.2079
7,COL17A1_Epi,2.00,0.9-4.42,0.087,2.4190,0.9983-5.8611,0.0504
8,GPX2_Epi,2.53,1.14-5.61,0.023,2.7510,1.2025-6.2932,0.0165


In [78]:
write.csv(out_2, './Survival_analysis/Output/R3-Q8/Epi_GSE26549_COX_res.csv', row.names = FALSE, na = "")

## TCGA

In [39]:
out_3 <- cox_pipe(epi_tcga, var_n = 13, filter_by_p_value = F, p_cutoff = 0.05)

In [40]:
out_3

,Characteristics,Hazard.Ratio,CI95,P.value,Hazard.Ratio.multi,CI95.multi,P.value.multi
,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,Stage_harmonized,1.71,1.15-2.53,0.008,1.8556,1.2193-2.8239,0.0039
2,Gender_harmonized,0.72,0.54-0.96,0.025,0.8364,0.5852-1.1953,0.3268
3,Age_harmonized,1.35,1.02-1.78,0.035,1.2502,0.9101-1.7174,0.1680
4,Smoking_harmonized,1.08,0.77-1.52,0.640,0.9408,0.6365-1.3905,0.7594
5,Alcohol_harmonized,0.92,0.69-1.22,0.563,0.9824,0.7051-1.3686,0.9162
6,HPV_status_harmonized,0.43,0.26-0.71,0.001,0.6207,0.3333-1.1559,0.1328
7,MKI67_Epi,1.37,1.01-1.85,0.042,1.6295,1.158-2.2931,0.0051
8,GPX2_Epi,1.43,1.09-1.88,0.009,1.4954,1.0812-2.0681,0.0150
9,COL17A1_Epi,1.46,1.11-1.92,0.007,1.5935,1.1214-2.2642,0.0093


In [79]:
write.csv(out_3, './Survival_analysis/Output/R3-Q8/Epi_TCGA_COX_res.csv', row.names = FALSE, na = "")

# 成纤维细胞的3个数据集csv输入

In [ ]:
fibro_geo_meta <- read.csv('./Survival_analysis/Input/Cox_input_R3_Q8/Fibro/GEO-meta/merged.uni_multi_cox_input.csv', stringsAsFactors = F)

In [ ]:
fibro_gse26549 <- read.csv('./Survival_analysis/Input/Cox_input_R3_Q8/Fibro/GSE26549/merged.uni_multi_cox_input.csv', stringsAsFactors = F)

In [ ]:
fibro_tcga <- read.csv('./Survival_analysis/Input/Cox_input_R3_Q8/Fibro/TCGA/merged.uni_multi_cox_input.csv', stringsAsFactors = F)

## GEO-meta

In [52]:
fibro_out_1 <- cox_pipe(fibro_geo_meta, var_n = 9, filter_by_p_value = F, p_cutoff = 0.05)

In [53]:
fibro_out_1

,Characteristics,Hazard.Ratio,CI95,P.value,Hazard.Ratio.multi,CI95.multi,P.value.multi
,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,Stage_harmonized,1.61,1.22-2.11,0.001,1.5268,1.154-2.0201,0.0030
2,Gender_harmonized,1.17,0.87-1.56,0.304,1.1834,0.8814-1.589,0.2627
3,Age_harmonized,0.88,0.69-1.13,0.322,0.9109,0.7137-1.1626,0.4536
4,Myofibroblast,1.65,1.3-2.1,0.000,1.7996,1.4053-2.3046,0.0000
5,CARBP1_Fibro,0.72,0.56-0.93,0.011,0.8203,0.6019-1.1179,0.2098
6,CFD_Fibro,0.70,0.54-0.91,0.007,0.7098,0.5214-0.9662,0.0294


In [80]:
write.csv(fibro_out_1, './Survival_analysis/Output/R3-Q8/Fibro_GEO-meta_COX_res.csv', row.names = FALSE, na = "")

## GSE26549

In [59]:
fibro_out_2 <- cox_pipe(fibro_gse26549, var_n = 10, filter_by_p_value = F, p_cutoff = 0.05)

In [60]:
fibro_out_2

,Characteristics,Hazard.Ratio,CI95,P.value,Hazard.Ratio.multi,CI95.multi,P.value.multi
,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,Gender_harmonized,0.95,0.48-1.86,0.872,0.8487,0.3969-1.8145,0.6721
2,Age_harmonized,1.55,0.79-3.06,0.203,1.4538,0.7146-2.9576,0.3018
3,Smoking_harmonized,1.55,0.72-3.32,0.263,1.6409,0.6995-3.8492,0.2550
4,Alcohol_harmonized,1.03,0.5-2.11,0.935,0.7500,0.3174-1.7726,0.5122
5,Myofibroblast,4.38,1.54-12.45,0.006,4.3030,1.4887-12.4371,0.0070
6,CARBP1_Fibro,0.49,0.25-0.95,0.035,0.5678,0.2367-1.362,0.2048
7,CFD_Fibro,0.47,0.23-0.94,0.034,0.6201,0.2426-1.5846,0.3181


In [81]:
write.csv(fibro_out_2, './Survival_analysis/Output/R3-Q8/Fibro_GSE26549_COX_res.csv', row.names = FALSE, na = "")

## TCGA

In [66]:
fibro_out_3 <- cox_pipe(fibro_tcga, var_n = 12, filter_by_p_value = F, p_cutoff = 0.05)

In [67]:
fibro_out_3

,Characteristics,Hazard.Ratio,CI95,P.value,Hazard.Ratio.multi,CI95.multi,P.value.multi
,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,Stage_harmonized,1.71,1.15-2.53,0.008,1.8884,1.2462-2.8615,0.0027
2,Gender_harmonized,0.72,0.54-0.96,0.025,0.8233,0.5775-1.1738,0.2827
3,Age_harmonized,1.35,1.02-1.78,0.035,1.2439,0.9051-1.7096,0.1785
4,Smoking_harmonized,1.08,0.77-1.52,0.640,1.0999,0.7461-1.6215,0.6306
5,Alcohol_harmonized,0.92,0.69-1.22,0.563,0.9213,0.6585-1.289,0.6322
6,HPV_status_harmonized,0.43,0.26-0.71,0.001,0.7073,0.3769-1.3272,0.2808
7,Myofibroblast,1.67,1.23-2.27,0.001,1.8876,1.2758-2.7929,0.0015
8,CFD_Fibro,0.76,0.58-1,0.047,0.8941,0.5943-1.345,0.5910
9,CARBP1_Fibro,0.77,0.58-1.01,0.063,0.6775,0.444-1.0338,0.0710


In [82]:
write.csv(fibro_out_3, './Survival_analysis/Output/R3-Q8/Fibro_TCGA_COX_res.csv', row.names = FALSE, na = "")

# 绘制森林图

In [ ]:
plot_cox_forest <- function(cox_df,
                            out_dir = ".",
                            title_tag = "cox",
                            var_order = NULL  
                            ) {
  library(forestplot)
  library(stringr)
  library(grid)

  needed_cols <- c("Characteristics",
                   "Hazard.Ratio", "CI95", "P.value",
                   "Hazard.Ratio.multi", "CI95.multi", "P.value.multi")
  if (!all(needed_cols %in% colnames(cox_df))) {
    stop("cox_df 缺少必要列，请确认列名为：\n",
         paste(needed_cols, collapse = ", "))
  }

  if (!dir.exists(out_dir)) {
    dir.create(out_dir, recursive = TRUE)
  }

  if (!is.null(var_order)) {
    not_in_df <- setdiff(var_order, cox_df$Characteristics)
    if (length(not_in_df) > 0) {
      warning("var_order 中这些变量不在 cox_df$Characteristics 中：",
              paste(not_in_df, collapse = ", "))
    }
    all_char <- c(var_order, setdiff(cox_df$Characteristics, var_order))
    cox_df <- cox_df[match(all_char, cox_df$Characteristics), ]
  } else {
    if ("Risk" %in% cox_df$Characteristics) {
      cox_df <- cox_df[order(cox_df$Characteristics == "Risk"), ]
    }
  }

  ci_uni   <- str_split_fixed(cox_df$CI95, "-", 2)
  ci_multi <- str_split_fixed(cox_df$CI95.multi, "-", 2)

  cox_df$Low.uni    <- as.numeric(ci_uni[, 1])
  cox_df$High.uni   <- as.numeric(ci_uni[, 2])
  cox_df$Low.multi  <- as.numeric(ci_multi[, 1])
  cox_df$High.multi <- as.numeric(ci_multi[, 2])

  .calc_xticks <- function(low, high) {
      vals <- c(low, high)
      vals <- vals[is.finite(vals)]
    
      rng <- range(vals, na.rm = TRUE)
      rng[1] <- max(floor(rng[1]), 0)
      rng[2] <- ceiling(rng[2])
    
      ticks <- seq(rng[1], rng[2], by = 1)
      if (!1 %in% ticks) ticks <- sort(c(ticks, 1))
      ticks
    }

  xticks_uni   <- .calc_xticks(cox_df$Low.uni,  cox_df$High.uni)
  xticks_multi <- .calc_xticks(cox_df$Low.multi, cox_df$High.multi)

  tabletext_uni <- cbind(
    c("Variable", cox_df$Characteristics),
    c(
      "HR (95% CI)",
      paste0(
        sprintf("%.2f", cox_df$Hazard.Ratio), " (",
        sprintf("%.2f", cox_df$Low.uni), "-",
        sprintf("%.2f", cox_df$High.uni), ")"
      )
    ),
    c(
      "P value",
      ifelse(cox_df$P.value < 0.001,
             "<0.001",
             sprintf("%.3f", cox_df$P.value))
    )
  )

  last_line_uni <- nrow(tabletext_uni) + 1
  hrzl_lines_uni <- list(
    "2" = gpar(lwd = 2, col = "black")
  )
  hrzl_lines_uni[[as.character(last_line_uni)]] <- gpar(lwd = 2, col = "black")

  tabletext_multi <- cbind(
    c("Variable", cox_df$Characteristics),
    c(
      "HR (95% CI)",
      paste0(
        sprintf("%.2f", cox_df$Hazard.Ratio.multi), " (",
        sprintf("%.2f", cox_df$Low.multi), "-",
        sprintf("%.2f", cox_df$High.multi), ")"
      )
    ),
    c(
      "P value",
      ifelse(cox_df$P.value.multi < 0.001,
             "<0.001",
             sprintf("%.3f", cox_df$P.value.multi))
    )
  )

  last_line_multi <- nrow(tabletext_multi) + 1
  hrzl_lines_multi <- list(
    "2" = gpar(lwd = 2, col = "black")
  )
  hrzl_lines_multi[[as.character(last_line_multi)]] <- gpar(lwd = 2, col = "black")

  pdf_file_uni <- file.path(out_dir, paste0("forest_uni_", title_tag, ".pdf"))
  pdf(pdf_file_uni, width = 12, height = 12,
      useDingbats = FALSE,
      family = "Helvetica")

  p_uni <- forestplot(
    labeltext = tabletext_uni,
    mean  = c(NA, cox_df$Hazard.Ratio),
    lower = c(NA, cox_df$Low.uni),
    upper = c(NA, cox_df$High.uni),

    graph.pos   = 2,
    graphwidth  = unit(.15, "npc"),
    fn.ci_norm  = "fpDrawCircleCI",
    col = fpColors(
      box  = "steelblue",
      lines = "black",
      zero  = "black"
    ),
    boxsize = 0.2,

    lwd.ci = 2,
    ci.vertices.height = 0.1,
    ci.vertices = TRUE,
    lwd.zero = 2,
    grid = structure(
      1, 
      gp = gpar(col = "black", lty = 2, lwd = 2)
    ),
    xticks   = xticks_uni,
    lwd.xaxis = 2,
    xlab = "Hazard Ratio (Univariate Cox)",

    hrzl_lines = hrzl_lines_uni,

    txt_gp = fpTxtGp(
      label = gpar(cex = 1),
      ticks = gpar(cex = 1),
      xlab  = gpar(cex = 1),
      title = gpar(cex = 1)
    ),
    lineheight = unit(0.75, "cm"),
    colgap     = unit(0, "cm"),
    mar        = unit(rep(1.25, times = 6), "cm"),
    new_page   = FALSE
  )
  print(p_uni)
  dev.off()
  message("✅ 已保存单因素森林图：", pdf_file_uni)

  pdf_file_multi <- file.path(out_dir, paste0("forest_multi_", title_tag, ".pdf"))
  pdf(pdf_file_multi, width = 12, height = 12,
      useDingbats = FALSE,
      family = "Helvetica")

  p_multi <- forestplot(
    labeltext = tabletext_multi,
    mean  = c(NA, cox_df$Hazard.Ratio.multi),
    lower = c(NA, cox_df$Low.multi),
    upper = c(NA, cox_df$High.multi),

    graph.pos   = 2,
    graphwidth  = unit(.15, "npc"),
    fn.ci_norm  = "fpDrawCircleCI",
    col = fpColors(
      box  = "steelblue",
      lines = "black",
      zero  = "black"
    ),
    boxsize = 0.2,

    lwd.ci = 2,
    ci.vertices.height = 0.1,
    ci.vertices = TRUE,
    lwd.zero = 2,
    grid = structure(
      1,
      gp = gpar(col = "black", lty = 2, lwd = 2)
    ),
    xticks   = xticks_multi,
    lwd.xaxis = 2,
    xlab = "Hazard Ratio (Multivariate Cox)",

    hrzl_lines = hrzl_lines_multi,

    txt_gp = fpTxtGp(
      label = gpar(cex = 1),
      ticks = gpar(cex = 1),
      xlab  = gpar(cex = 1),
      title = gpar(cex = 1)
    ),
    lineheight = unit(0.75, "cm"),
    colgap     = unit(0, "cm"),
    mar        = unit(rep(1.25, times = 6), "cm"),
    new_page   = FALSE
  )

  print(p_multi)
  dev.off()
  message("✅ 已保存多因素森林图：", pdf_file_multi)
}


## Epi

In [94]:
out_1

,Characteristics,Hazard.Ratio,CI95,P.value,Hazard.Ratio.multi,CI95.multi,P.value.multi
,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,Stage_harmonized,1.61,1.22-2.11,0.001,1.6259,1.2309-2.1477,0.0006
2,Gender_harmonized,1.17,0.87-1.56,0.304,1.1099,0.8238-1.4955,0.4930
3,Age_harmonized,0.88,0.69-1.13,0.322,0.8734,0.6828-1.1173,0.2814
4,SBSN_Epi,0.77,0.6-0.98,0.030,0.8657,0.6784-1.1048,0.2464
5,COL17A1_Epi,1.69,1.29-2.22,0.000,1.8996,1.4393-2.507,0.0000
6,MKI67_Epi,1.57,1.21-2.03,0.001,1.5190,1.1619-1.9859,0.0022
7,GPX2_Epi,1.30,1.02-1.67,0.037,1.3079,1.0101-1.6936,0.0417


In [ ]:
plot_cox_forest(out_1, out_dir = './Survival_analysis/Output/R3-Q8/Epi/GEO-meta', 
                var_order = c('Stage_harmonized', 'Gender_harmonized', 'Age_harmonized', 'COL17A1_Epi', 'MKI67_Epi', 'GPX2_Epi', 'SBSN_Epi'))

✅ 已保存单因素森林图：./Survival_analysis/Output/R3-Q8/Epi/GEO-meta/forest_uni_cox.pdf

✅ 已保存多因素森林图：./Survival_analysis/Output/R3-Q8/Epi/GEO-meta/forest_multi_cox.pdf



In [96]:
out_2$Characteristics

[1] "Gender_harmonized"  "Age_harmonized"     "Smoking_harmonized"
[4] "Alcohol_harmonized" "MKI67_Epi"          "SBSN_Epi"          
[7] "COL17A1_Epi"        "GPX2_Epi"

In [ ]:
plot_cox_forest(out_2, out_dir = './Survival_analysis/Output/R3-Q8/Epi/GSE26549', 
                var_order = c('Gender_harmonized', 'Age_harmonized', 'Smoking_harmonized', 'Alcohol_harmonized', 'COL17A1_Epi', 'MKI67_Epi', 'GPX2_Epi', 'SBSN_Epi'))

✅ 已保存单因素森林图：./Survival_analysis/Output/R3-Q8/Epi/GSE26549/forest_uni_cox.pdf

✅ 已保存多因素森林图：./Survival_analysis/Output/R3-Q8/Epi/GSE26549/forest_multi_cox.pdf



In [98]:
out_3$Characteristics

[1] "Stage_harmonized"      "Gender_harmonized"     "Age_harmonized"       
 [4] "Smoking_harmonized"    "Alcohol_harmonized"    "HPV_status_harmonized"
 [7] "MKI67_Epi"             "GPX2_Epi"              "COL17A1_Epi"          
[10] "SBSN_Epi"

In [ ]:
plot_cox_forest(out_3, out_dir = './Survival_analysis/Output/R3-Q8/Epi/TCGA', 
                var_order = c('Stage_harmonized', 'Gender_harmonized', 'Age_harmonized', 'Smoking_harmonized', 'Alcohol_harmonized', 'HPV_status_harmonized', 'COL17A1_Epi', 'MKI67_Epi', 'GPX2_Epi', 'SBSN_Epi'))

✅ 已保存单因素森林图：./Survival_analysis/Output/R3-Q8/Epi/TCGA/forest_uni_cox.pdf

✅ 已保存多因素森林图：./Survival_analysis/Output/R3-Q8/Epi/TCGA/forest_multi_cox.pdf



## Fibro

In [100]:
fibro_out_1$Characteristics

[1] "Stage_harmonized"  "Gender_harmonized" "Age_harmonized"   
[4] "Myofibroblast"     "CARBP1_Fibro"      "CFD_Fibro"

In [101]:
plot_cox_forest(fibro_out_1, out_dir = './Survival_analysis/Output/R3-Q8/Fibro/GEO-meta', 
                var_order = c('Stage_harmonized', 'Gender_harmonized', 'Age_harmonized', 'CFD_Fibro', 'CARBP1_Fibro', 'Myofibroblast'))

✅ 已保存单因素森林图：./Survival_analysis/Output/R3-Q8/Fibro/GEO-meta/forest_uni_cox.pdf

✅ 已保存多因素森林图：./Survival_analysis/Output/R3-Q8/Fibro/GEO-meta/forest_multi_cox.pdf



In [91]:
fibro_out_2$Characteristics

[1] "Gender_harmonized"  "Age_harmonized"     "Smoking_harmonized"
[4] "Alcohol_harmonized" "Myofibroblast"      "CARBP1_Fibro"      
[7] "CFD_Fibro"

In [102]:
plot_cox_forest(fibro_out_2, out_dir = './Survival_analysis/Output/R3-Q8/Fibro/GSE26549', 
                var_order = c('Gender_harmonized', 'Age_harmonized', 'Smoking_harmonized', 'Alcohol_harmonized', 'CFD_Fibro', 'CARBP1_Fibro', 'Myofibroblast'))

✅ 已保存单因素森林图：./Survival_analysis/Output/R3-Q8/Fibro/GSE26549/forest_uni_cox.pdf

✅ 已保存多因素森林图：./Survival_analysis/Output/R3-Q8/Fibro/GSE26549/forest_multi_cox.pdf



In [103]:
fibro_out_3$Characteristics

[1] "Stage_harmonized"      "Gender_harmonized"     "Age_harmonized"       
[4] "Smoking_harmonized"    "Alcohol_harmonized"    "HPV_status_harmonized"
[7] "Myofibroblast"         "CFD_Fibro"             "CARBP1_Fibro"

In [104]:
plot_cox_forest(fibro_out_3, out_dir = './Survival_analysis/Output/R3-Q8/Fibro/TCGA', 
                var_order = c('Stage_harmonized', 'Gender_harmonized', 'Age_harmonized', 'Smoking_harmonized', 'Alcohol_harmonized', 'HPV_status_harmonized', 'CFD_Fibro', 'CARBP1_Fibro', 'Myofibroblast'))

✅ 已保存单因素森林图：./Survival_analysis/Output/R3-Q8/Fibro/TCGA/forest_uni_cox.pdf

✅ 已保存多因素森林图：./Survival_analysis/Output/R3-Q8/Fibro/TCGA/forest_multi_cox.pdf

